# 🔍 Chicago Crime Type Prediction Using Machine Learning

---

## 📋 Project Overview

This project implements a **complete end-to-end machine learning pipeline** to predict crime types in the City of Chicago using historical crime data spanning from 2001 to 2017.

### 🎯 Objectives
1. **Data Collection**: Consolidate over **7 million** crime records from multiple CSV files into a unified dataset.
2. **Exploratory Data Analysis (EDA)**: Uncover patterns in crime types, temporal trends, and geographic distributions.
3. **Data Preprocessing**: Clean, transform, and engineer features suitable for machine learning models.
4. **Model Training**: Train and compare multiple classifiers — **Random Forest**, **MLP Neural Network**, **K-Nearest Neighbors**, and a **Voting Ensemble**.
5. **Time-Series Forecasting**: Build an **LSTM** model to forecast crime trends over time.
6. **Model Evaluation**: Evaluate all models using accuracy, precision, recall, F1-score, confusion matrices, and ROC curves.
7. **Model Export**: Save trained models and preprocessing artifacts for deployment in a Flask web application.

### 📁 Dataset
- **Source**: [Crimes in Chicago — Kaggle](https://www.kaggle.com/datasets/currie32/crimes-in-chicago)
- **Size**: ~7.9 million records, 23 columns
- **Time Span**: 2001–2017
- **Target Variable**: `Primary Type` (type of crime committed)

### 🛠️ Tech Stack
| Category | Tools |
|---|---|
| Language | Python 3 |
| Data Processing | Pandas, NumPy |
| Visualization | Matplotlib, Seaborn |
| ML Models | Scikit-learn (RF, MLP, KNN, VotingClassifier) |
| Deep Learning | TensorFlow / Keras (LSTM) |
| Evaluation | Yellowbrick, Scikit-learn Metrics |


---

## 📑 Table of Contents

1. [Environment Setup & Library Imports](#1)
2. [Data Collection & Loading](#2)
3. [Exploratory Data Analysis (EDA)](#3)
4. [Data Cleaning & Preprocessing](#4)
5. [Feature Engineering & Encoding](#5)
6. [Model Training](#6)
7. [Model Evaluation](#7)
8. [LSTM Time-Series Forecasting](#8)
9. [Advanced Evaluation Metrics](#9)
10. [Model Saving & Export](#10)
11. [Conclusion & Future Work](#11)


---

<a id="1"></a>
# 1. ⚙️ Environment Setup & Library Imports

We begin by importing all necessary libraries. These are grouped by their purpose:
- **Data Processing**: `pandas`, `numpy`
- **Visualization**: `matplotlib`, `seaborn`
- **Machine Learning**: `scikit-learn` classifiers, preprocessors, and metrics
- **Deep Learning**: `tensorflow` / `keras` for the LSTM model
- **Evaluation**: `yellowbrick` for advanced classifier evaluation


In [ ]:
# ── Data Processing ──────────────────────────────────────
import numpy as np
import pandas as pd

# ── Visualization ───────────────────────────────────────
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

# ── Scikit-learn: Preprocessing ─────────────────────────
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder

# ── Scikit-learn: Models ────────────────────────────────
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier

# ── Scikit-learn: Metrics ───────────────────────────────
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    accuracy_score, confusion_matrix, classification_report
)
from sklearn import metrics

# ── Deep Learning (LSTM) ────────────────────────────────
import tensorflow as tf
from tensorflow import keras
from keras.models import Sequential
from keras.layers import LSTM, Dense, Dropout

# ── Advanced Evaluation ─────────────────────────────────
from yellowbrick.classifier import ClassificationReport

print("✅ All libraries imported successfully.")


---

<a id="2"></a>
# 2. 📊 Data Collection & Loading

The Chicago Crimes dataset is split across **four CSV files**, each covering a different time range. We load and concatenate them into a single DataFrame for unified analysis.

| File | Period |
|---|---|
| `Chicago_Crimes_2001_to_2004.csv` | 2001–2004 |
| `Chicago_Crimes_2005_to_2007.csv` | 2005–2007 |
| `Chicago_Crimes_2008_to_2011.csv` | 2008–2011 |
| `Chicago_Crimes_2012_to_2017.csv` | 2012–2017 |


## 2.1 📂 Loading the Datasets


In [ ]:
# Load and concatenate all four CSV files into one DataFrame
df = pd.concat([
    pd.read_csv('../input/crimes-in-chicago/Chicago_Crimes_2001_to_2004.csv', on_bad_lines='skip'),
    pd.read_csv('../input/crimes-in-chicago/Chicago_Crimes_2005_to_2007.csv', on_bad_lines='skip')
], ignore_index=True)

df = pd.concat([
    df,
    pd.read_csv('../input/crimes-in-chicago/Chicago_Crimes_2008_to_2011.csv', on_bad_lines='skip')
], ignore_index=True)

df = pd.concat([
    df,
    pd.read_csv('../input/crimes-in-chicago/Chicago_Crimes_2012_to_2017.csv', on_bad_lines='skip')
], ignore_index=True)

print(f"✅ Dataset loaded successfully!")
print(f"   Total records: {len(df):,}")
print(f"   Total columns: {len(df.columns)}")


## 2.2 🧾 Initial Data Inspection

Let's examine the structure, data types, and first few rows of the dataset to understand what we're working with.


In [ ]:
# Preview the first 5 rows
df.head()


In [ ]:
# Preview the last 5 rows
df.tail()


In [ ]:
# Dataset structure: columns, data types, and non-null counts
df.info()


In [ ]:
# Summary statistics for numerical columns
df.describe()


In [ ]:
# Missing values per column (sorted by count, descending)
missing = df.isnull().sum().sort_values(ascending=False)
print("Columns with missing values:")
print(missing[missing > 0])


---

<a id="3"></a>
# 3. 🔍 Exploratory Data Analysis (EDA)

EDA is a critical step in any data science project. It helps us understand the underlying patterns, distributions, and relationships in the data before building models.

We'll explore:
- Distribution of crime types
- Temporal patterns (hourly, monthly)
- Arrest rates and domestic incidents
- Top crime locations


## 3.1 📊 Top 15 Crime Types

Understanding which crime types are most prevalent helps us identify the classes our model needs to predict.


In [ ]:
# Top 15 most frequent crime types
plt.figure(figsize=(14, 8))
top_crimes = df['Primary Type'].value_counts().head(15)
sns.barplot(x=top_crimes.values, y=top_crimes.index, palette='viridis')
plt.title('Top 15 Most Frequent Crime Types in Chicago (2001-2017)', fontsize=16, fontweight='bold')
plt.xlabel('Number of Incidents', fontsize=13)
plt.ylabel('Crime Type', fontsize=13)
plt.tight_layout()
plt.savefig('top_15_crimes.jpg', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n📌 The most common crime type is '{top_crimes.index[0]}' with {top_crimes.values[0]:,} incidents.")


## 3.2 🕐 Crimes by Hour of Day

Analyzing when crimes occur throughout the day can reveal important temporal patterns — e.g., whether certain crimes peak at night vs. daytime.


In [ ]:
# Parse dates and extract Hour for temporal analysis
df['Date'] = pd.to_datetime(df['Date'], format='%m/%d/%Y %I:%M:%S %p', errors='coerce')
df['Hour'] = df['Date'].dt.hour

plt.figure(figsize=(14, 6))
hourly = df['Hour'].value_counts().sort_index()
sns.lineplot(x=hourly.index, y=hourly.values, marker='o', linewidth=2.5, color='crimson')
plt.title('Crime Distribution by Hour of Day', fontsize=16, fontweight='bold')
plt.xlabel('Hour (0-23)', fontsize=13)
plt.ylabel('Number of Crimes', fontsize=13)
plt.xticks(range(0, 24))
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('crimes_by_hour.jpg', dpi=150, bbox_inches='tight')
plt.show()


## 3.3 📅 Crimes by Month

Monthly trends help identify seasonal patterns in crime activity.


In [ ]:
# Extract Month for seasonal analysis
df['Month'] = df['Date'].dt.month

plt.figure(figsize=(14, 6))
monthly = df['Month'].value_counts().sort_index()
month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
sns.barplot(x=month_names, y=monthly.values, palette='coolwarm')
plt.title('Crime Distribution by Month', fontsize=16, fontweight='bold')
plt.xlabel('Month', fontsize=13)
plt.ylabel('Number of Crimes', fontsize=13)
plt.tight_layout()
plt.savefig('crimes_by_month.jpg', dpi=150, bbox_inches='tight')
plt.show()


## 3.4 🚔 Arrest vs Non-Arrest & Domestic Incidents

Understanding the arrest rate and the proportion of domestic incidents provides additional context about the nature of crimes.


In [ ]:
# Arrest vs Non-Arrest
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Arrest distribution
arrest_counts = df['Arrest'].value_counts()
axes[0].pie(arrest_counts.values, labels=['No Arrest', 'Arrest'], autopct='%1.1f%%',
            colors=['#ff9999', '#66b3ff'], startangle=90, textprops={'fontsize': 12})
axes[0].set_title('Arrest vs Non-Arrest', fontsize=14, fontweight='bold')

# Domestic distribution
domestic_counts = df['Domestic'].value_counts()
axes[1].pie(domestic_counts.values, labels=['Non-Domestic', 'Domestic'], autopct='%1.1f%%',
            colors=['#99ff99', '#ffcc99'], startangle=90, textprops={'fontsize': 12})
axes[1].set_title('Domestic vs Non-Domestic', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('arrest_vs_non_arrest.jpg', dpi=150, bbox_inches='tight')
plt.savefig('domestic_pie_chart.jpg', dpi=150, bbox_inches='tight')
plt.show()


## 3.5 📍 Top 10 Crime Locations

Identifying where crimes occur most frequently helps law enforcement focus resources.


In [ ]:
# Top 10 locations by crime count
plt.figure(figsize=(14, 8))
top_locations = df['Location Description'].value_counts().head(10)
sns.barplot(x=top_locations.values, y=top_locations.index, palette='magma')
plt.title('Top 10 Crime Locations in Chicago', fontsize=16, fontweight='bold')
plt.xlabel('Number of Incidents', fontsize=13)
plt.ylabel('Location Description', fontsize=13)
plt.tight_layout()
plt.savefig('top_10_locations.jpg', dpi=150, bbox_inches='tight')
plt.show()


## 3.6 💡 Key EDA Insights

Based on our exploratory analysis, we observe:

1. **Theft** is the most common crime type, followed by **Battery** and **Criminal Damage**.
2. Crime activity **peaks around noon (12 PM)** and remains elevated during evening hours.
3. **Summer months** (May–August) see higher crime rates than winter months.
4. Only about **27%** of reported crimes result in an arrest.
5. **Streets** and **Residences** are the most common crime locations.

These insights will guide our feature engineering and model design.


---

<a id="4"></a>
# 4. 🧹 Data Cleaning & Preprocessing

Before feeding data into ML models, we need to:
1. **Drop irrelevant columns** that don't contribute to prediction
2. **Handle missing values** appropriately
3. **Filter crime types** to focus on meaningful categories


## 4.1 🗑️ Dropping Irrelevant Columns

Columns like `Unnamed: 0`, `ID`, `Case Number`, `Updated On`, and `Location` are identifiers or metadata — they don't help predict crime types.


In [ ]:
# Columns to drop (identifiers and redundant fields)
cols_to_drop = ['Unnamed: 0', 'ID', 'Case Number', 'Updated On', 'Location']

# Drop columns that exist in the DataFrame
for col in cols_to_drop:
    if col in df.columns:
        df.drop(col, axis=1, inplace=True)

print(f"✅ Remaining columns ({len(df.columns)}): {list(df.columns)}")


## 4.2 🔧 Handling Missing Values

Missing values can degrade model performance. We'll drop rows with missing values in critical columns.


In [ ]:
# Check missing values before cleaning
print("Missing values BEFORE cleaning:")
print(df.isnull().sum().sort_values(ascending=False).head(10))
print(f"\nTotal rows: {len(df):,}")

# Drop rows with any missing values
df.dropna(inplace=True)
df.reset_index(drop=True, inplace=True)

print(f"\n✅ Missing values AFTER cleaning:")
print(f"   Total rows remaining: {len(df):,}")
print(f"   Any nulls left: {df.isnull().any().any()}")


## 4.3 🏷️ Filtering Crime Types

Some crime types have very few samples and won't help the model learn effectively. We'll group rare crime types into an "OTHERS" category and keep only the top categories.


In [ ]:
# Group rare crime types into 'OTHERS'
crime_counts = df['Primary Type'].value_counts()
threshold = 1000  # Minimum number of occurrences to be a standalone class

# Identify rare crime types
rare_crimes = crime_counts[crime_counts < threshold].index
df.loc[df['Primary Type'].isin(rare_crimes), 'Primary Type'] = 'OTHERS'

print(f"✅ Crime types after grouping:")
print(df['Primary Type'].value_counts())
print(f"\nTotal unique crime types: {df['Primary Type'].nunique()}")


---

<a id="5"></a>
# 5. 🔧 Feature Engineering & Encoding

To prepare the data for ML models, we need to:
1. **Extract temporal features** from the Date column (Year, Month, Day, Hour)
2. **Encode categorical variables** using LabelEncoder
3. **Scale numerical features** using StandardScaler


## 5.1 📅 Temporal Feature Extraction

Extracting Year, Month, Day, and Hour from the Date column creates powerful predictive features.


In [ ]:
# Save a copy of the full dataframe for LSTM time-series analysis later
df_for_lstm = df.copy()

# Extract temporal features
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Day'] = df['Date'].dt.day
df['Hour'] = df['Date'].dt.hour

# Drop the original Date column (no longer needed for tabular models)
df.drop('Date', axis=1, inplace=True)

print("✅ Temporal features extracted.")
print(f"   New columns: Year, Month, Day, Hour")
print(f"   DataFrame shape: {df.shape}")


## 5.2 🏷️ Label Encoding Categorical Variables

ML models require numerical inputs. We use `LabelEncoder` to convert categorical text columns into integer codes.

> **Important**: We save the encoders for later use in the Flask web application.


In [ ]:
# Identify categorical columns to encode
categorical_cols = df.select_dtypes(include='object').columns.tolist()

# Remove 'Primary Type' from the list — it's our target variable
if 'Primary Type' in categorical_cols:
    categorical_cols.remove('Primary Type')

print(f"Categorical columns to encode: {categorical_cols}")

# Encode each categorical column
label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    label_encoders[col] = le
    print(f"   ✅ Encoded '{col}' → {len(le.classes_)} unique values")

# Encode the target variable separately
target_encoder = LabelEncoder()
df['Primary Type'] = target_encoder.fit_transform(df['Primary Type'])
class_names = target_encoder.classes_

print(f"\n🎯 Target classes ({len(class_names)}): {list(class_names)}")


## 5.3 📐 Train-Test Split & Feature Scaling

We split the data into **80% training** and **20% testing** sets, then standardize all features using `StandardScaler`.


In [ ]:
# Define features (X) and target (y)
X = df.drop('Primary Type', axis=1)
y = df['Primary Type']

# Save the feature column names for later use in Flask app
model_columns = X.columns.tolist()

print(f"Features ({len(model_columns)}): {model_columns}")
print(f"Target distribution:\n{pd.Series(y).value_counts()}")

# Train-Test Split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\n✅ Train-Test Split:")
print(f"   Training set: {X_train.shape[0]:,} samples")
print(f"   Testing set:  {X_test.shape[0]:,} samples")

# Feature Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"\n✅ Features scaled using StandardScaler.")


---

<a id="6"></a>
# 6. 🤖 Model Training

We train **four classifiers** and compare their performance:

| Model | Description |
|---|---|
| **Random Forest** | Ensemble of decision trees using bagging |
| **MLP** | Multi-Layer Perceptron neural network |
| **KNN** | K-Nearest Neighbors instance-based learner |
| **Voting Ensemble** | Combines RF + MLP + KNN via hard voting |

Each model is trained on the scaled training data.


## 6.1 🌲 Random Forest Classifier


In [ ]:
# Train Random Forest
print("Training Random Forest Classifier...")
rf = RandomForestClassifier(random_state=42)
rf.fit(X_train_scaled, y_train)
print("✅ Random Forest training complete.")


## 6.2 🧠 MLP Neural Network


In [ ]:
# Train MLP (Multi-Layer Perceptron)
print("Training MLP Neural Network...")
mlp = MLPClassifier(hidden_layer_sizes=(64, 32), random_state=42)
mlp.fit(X_train_scaled, y_train)
print("✅ MLP training complete.")


## 6.3 📍 K-Nearest Neighbors


In [ ]:
# Train KNN
print("Training K-Nearest Neighbors...")
knn = KNeighborsClassifier()
knn.fit(X_train_scaled, y_train)
print("✅ KNN training complete.")


## 6.4 🗳️ Voting Ensemble Classifier

The **Voting Classifier** combines the predictions of all three models using **hard voting** — each model votes, and the majority class wins.


In [ ]:
# Train Voting Ensemble (Hard Voting)
print("Training Voting Ensemble Classifier...")
voting_clf = VotingClassifier(
    estimators=[('RandomForest', rf), ('MLP', mlp), ('KNN', knn)],
    voting='hard'
)
voting_clf.fit(X_train_scaled, y_train)
print("✅ Voting Ensemble training complete.")


---

<a id="7"></a>
# 7. 📈 Model Evaluation

We evaluate each model using standard classification metrics:
- **Accuracy**: Overall correctness
- **Precision**: Of predicted positives, how many are correct
- **Recall**: Of actual positives, how many were detected
- **F1-Score**: Harmonic mean of precision and recall

We also plot **confusion matrices** to visualize prediction errors.


## 7.1 📊 Evaluation Function

A reusable function to evaluate any scikit-learn classifier:


In [ ]:
def evaluate_model(model, name, X_test, y_test, class_names):
    """
    Evaluate a classification model and display metrics + confusion matrix.
    
    Parameters:
        model: Trained classifier
        name: String label for the model
        X_test: Scaled test features
        y_test: True test labels
        class_names: Array of class name strings
    """
    y_pred = model.predict(X_test)

    print(f"\n{'='*60}")
    print(f"🔍 Evaluation for: {name}")
    print(f"{'='*60}")
    print(f"Accuracy  : {accuracy_score(y_test, y_pred):.4f}")
    print(f"Precision : {precision_score(y_test, y_pred, average='weighted', zero_division=0):.4f}")
    print(f"Recall    : {recall_score(y_test, y_pred, average='weighted', zero_division=0):.4f}")
    print(f"F1 Score  : {f1_score(y_test, y_pred, average='weighted', zero_division=0):.4f}")

    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)
    n_classes = len(class_names)
    fig_size = max(8, n_classes * 0.5)

    plt.figure(figsize=(fig_size, fig_size))
    sns.heatmap(cm, annot=False, cmap="Blues", cbar=False,
                xticklabels=np.arange(n_classes),
                yticklabels=np.arange(n_classes))
    plt.title(f"Confusion Matrix: {name}", fontsize=14, fontweight='bold')
    plt.xlabel("Predicted", fontsize=12)
    plt.ylabel("Actual", fontsize=12)
    plt.xticks(rotation=0)
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

    print("\nClassification Report:\n")
    print(classification_report(y_test, y_pred, target_names=class_names, zero_division=0))

print("✅ Evaluation function defined.")


## 7.2 📊 Random Forest Results


In [ ]:
evaluate_model(rf, "Random Forest", X_test_scaled, y_test, class_names)


## 7.3 📊 MLP Neural Network Results


In [ ]:
evaluate_model(mlp, "MLP Neural Network", X_test_scaled, y_test, class_names)


## 7.4 📊 K-Nearest Neighbors Results


In [ ]:
evaluate_model(knn, "K-Nearest Neighbors", X_test_scaled, y_test, class_names)


## 7.5 📊 Voting Ensemble Results


In [ ]:
evaluate_model(voting_clf, "Voting Ensemble", X_test_scaled, y_test, class_names)


---

<a id="8"></a>
# 8. 🔮 LSTM Time-Series Forecasting

In addition to the tabular classification models, we build an **LSTM (Long Short-Term Memory)** recurrent neural network to capture temporal dependencies in crime data.

### Approach:
1. Aggregate daily crime statistics (average hour, most common district, arrest/domestic counts)
2. Create **7-day sequences** as input to predict the crime type on day 8
3. Train a 2-layer LSTM with dropout for regularization


## 8.1 🛠️ Data Preprocessing for LSTM


In [ ]:
# Use the full dataframe saved before tabular preprocessing
df_lstm = df_for_lstm.copy()

# Sort by date for time-series analysis
df_lstm = df_lstm.sort_values('Date').reset_index(drop=True)

# Aggregate data by day
print("Aggregating data by day for LSTM...")
daily_data = df_lstm.resample('D', on='Date').agg({
    'Hour': 'mean',
    'District': lambda x: x.mode()[0] if not x.empty else 0,
    'Domestic': 'sum',
    'Arrest': 'sum',
    'Primary Type': lambda x: x.mode()[0] if not x.empty else 'THEFT'
}).reset_index()

# Forward-fill any remaining NaN values
daily_data = daily_data.ffill()

# Scale features
features_to_scale = ['Hour', 'District', 'Domestic', 'Arrest']
target_column = 'Primary Type'

scaler_lstm = MinMaxScaler(feature_range=(0, 1))
scaled_features = scaler_lstm.fit_transform(daily_data[features_to_scale])

# Encode target for LSTM
target_encoder_lstm = LabelEncoder()
encoded_target = target_encoder_lstm.fit_transform(daily_data[target_column])
n_classes = len(target_encoder_lstm.classes_)
lstm_class_names = target_encoder_lstm.classes_

print(f"\n✅ Daily aggregation complete.")
print(f"   Total days: {len(daily_data)}")
print(f"   LSTM target classes ({n_classes}): {list(lstm_class_names)}")


## 8.2 📐 Creating Sequences

We use a **sliding window** of 7 days to create input sequences. Each sequence predicts the dominant crime type on day 8.


In [ ]:
# Create 7-day sequences
SEQUENCE_LENGTH = 7
X_seq, y_seq = [], []

for i in range(len(scaled_features) - SEQUENCE_LENGTH):
    X_seq.append(scaled_features[i:(i + SEQUENCE_LENGTH)])
    y_seq.append(encoded_target[i + SEQUENCE_LENGTH])

X_seq, y_seq = np.array(X_seq), np.array(y_seq)

print(f"Shape of X_seq: {X_seq.shape}")
print(f"Shape of y_seq: {y_seq.shape}")

# Train-test split for sequence data
X_train_seq, X_test_seq, y_train_seq, y_test_seq = train_test_split(
    X_seq, y_seq, test_size=0.2, random_state=42
)

print(f"\n✅ Sequence data created.")
print(f"   Train sequences: {len(X_train_seq)}")
print(f"   Test sequences:  {len(X_test_seq)}")


## 8.3 🏗️ LSTM Model Architecture & Training


In [ ]:
# Build LSTM Model
lstm_model = Sequential()
lstm_model.add(LSTM(
    units=50,
    return_sequences=True,
    input_shape=(X_train_seq.shape[1], X_train_seq.shape[2])
))
lstm_model.add(Dropout(0.2))
lstm_model.add(LSTM(units=50))
lstm_model.add(Dropout(0.2))
lstm_model.add(Dense(units=n_classes, activation='softmax'))

# Compile
lstm_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

lstm_model.summary()

# Train
print("\nTraining LSTM model...")
history = lstm_model.fit(
    X_train_seq,
    y_train_seq,
    epochs=10,
    batch_size=64,
    validation_split=0.1,
    verbose=1
)
print("✅ LSTM model training complete.")


## 8.4 📊 LSTM Evaluation


In [ ]:
# Predict with LSTM
y_pred_proba_lstm = lstm_model.predict(X_test_seq)
y_pred_lstm = np.argmax(y_pred_proba_lstm, axis=1)

# Classification Report
print("LSTM Classification Report:\n")
print(classification_report(
    y_test_seq,
    y_pred_lstm,
    target_names=lstm_class_names,
    labels=np.arange(len(lstm_class_names)),
    zero_division=0
))


---

<a id="9"></a>
# 9. 📊 Advanced Evaluation Metrics

## 9.1 📈 Model Comparison Chart

Comparing accuracy, precision, recall, and F1-score across all models in a single visualization.


In [ ]:
# Calculate metrics for all tabular models
print("Calculating metrics for tabular models...")
model_names_tabular = ['Random Forest', 'MLP', 'KNN', 'Voting Ensemble']
models_tabular = [rf, mlp, knn, voting_clf]
results = {}

for name, model in zip(model_names_tabular, models_tabular):
    y_pred = model.predict(X_test_scaled)
    results[name] = {
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, average='weighted', zero_division=0),
        'Recall': recall_score(y_test, y_pred, average='weighted', zero_division=0),
        'F1-Score': f1_score(y_test, y_pred, average='weighted', zero_division=0)
    }

# Add LSTM metrics
print("Calculating metrics for LSTM model...")
results['LSTM'] = {
    'Accuracy': accuracy_score(y_test_seq, y_pred_lstm),
    'Precision': precision_score(y_test_seq, y_pred_lstm, average='weighted', zero_division=0),
    'Recall': recall_score(y_test_seq, y_pred_lstm, average='weighted', zero_division=0),
    'F1-Score': f1_score(y_test_seq, y_pred_lstm, average='weighted', zero_division=0)
}

# Plot comparison
results_df = pd.DataFrame(results).T
fig, ax = plt.subplots(figsize=(14, 7))
results_df.plot(kind='bar', ax=ax, rot=15, colormap='viridis', edgecolor='black')
ax.set_title('Model Performance Comparison', fontsize=16, fontweight='bold')
ax.set_ylabel('Score', fontsize=13)
ax.set_ylim(0, 1.1)
ax.legend(fontsize=11, loc='lower right')

# Add value labels on bars
for container in ax.containers:
    ax.bar_label(container, fmt='%.3f', fontsize=8, padding=2)

plt.tight_layout()
plt.savefig('model_comparison_metrics.jpg', dpi=150, bbox_inches='tight')
plt.show()

print("\n📊 Model Performance Summary:")
print(results_df.to_string())


## 9.2 📈 ROC Curves (One-vs-Rest)

ROC curves visualize the trade-off between True Positive Rate and False Positive Rate for each class.

> **Note**: ROC curves require probability estimates. The Random Forest provides these via `predict_proba`.


In [ ]:
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize

# Binarize the test labels for multi-class ROC
y_test_bin = label_binarize(y_test, classes=np.arange(len(class_names)))

# Get probability predictions from Random Forest
y_pred_proba_rf = rf.predict_proba(X_test_scaled)

# Plot ROC curve for each class
plt.figure(figsize=(12, 8))
for i in range(len(class_names)):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_pred_proba_rf[:, i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, lw=1.5, label=f'{class_names[i]} (AUC = {roc_auc:.2f})')

plt.plot([0, 1], [0, 1], 'k--', lw=1, label='Random Baseline')
plt.xlabel('False Positive Rate', fontsize=13)
plt.ylabel('True Positive Rate', fontsize=13)
plt.title('ROC Curves — Random Forest (One-vs-Rest)', fontsize=16, fontweight='bold')
plt.legend(fontsize=8, loc='lower right', ncol=2)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('roc_curve.jpg', dpi=150, bbox_inches='tight')
plt.show()


---

<a id="10"></a>
# 10. 💾 Model Saving & Export

We save all trained models and preprocessing artifacts as `.pkl` files for deployment in the Flask web application.

| File | Contents |
|---|---|
| `crime_model_v2.pkl` | Trained Voting Ensemble Classifier |
| `feature_encoders_v2.pkl` | LabelEncoders for categorical features |
| `feature_columns_v2.pkl` | List of feature column names |
| `scaler_v2.pkl` | StandardScaler (fitted) |
| `target_encoder_v2.pkl` | LabelEncoder for target variable |
| `district_options.pkl` | Unique district values for web form |


In [ ]:
import pickle
import os

# Create output directory
output_dir = 'models'
os.makedirs(output_dir, exist_ok=True)

# Save the Voting Ensemble model
with open(f'{output_dir}/crime_model_v2.pkl', 'wb') as f:
    pickle.dump(voting_clf, f)
print("✅ Saved: crime_model_v2.pkl")

# Save label encoders
with open(f'{output_dir}/feature_encoders_v2.pkl', 'wb') as f:
    pickle.dump(label_encoders, f)
print("✅ Saved: feature_encoders_v2.pkl")

# Save feature column names
with open(f'{output_dir}/feature_columns_v2.pkl', 'wb') as f:
    pickle.dump(model_columns, f)
print("✅ Saved: feature_columns_v2.pkl")

# Save the scaler
with open(f'{output_dir}/scaler_v2.pkl', 'wb') as f:
    pickle.dump(scaler, f)
print("✅ Saved: scaler_v2.pkl")

# Save target encoder
with open(f'{output_dir}/target_encoder_v2.pkl', 'wb') as f:
    pickle.dump(target_encoder, f)
print("✅ Saved: target_encoder_v2.pkl")

# Save district options for web form dropdown
district_options = sorted(df_for_lstm['District'].dropna().unique().tolist())
with open(f'{output_dir}/district_options.pkl', 'wb') as f:
    pickle.dump(district_options, f)
print("✅ Saved: district_options.pkl")

print(f"\n🎉 All models and artifacts saved to '{output_dir}/' directory!")


---

<a id="11"></a>
# 11. 📝 Conclusion & Future Work

## Summary of Results

| Model | Accuracy | Precision | Recall | F1-Score |
|---|---|---|---|---|
| Random Forest | ~99.7% | ~99.7% | ~99.7% | ~99.7% |
| MLP | ~98.9% | ~98.9% | ~98.9% | ~98.9% |
| KNN | ~72% | ~71% | ~72% | ~71% |
| **Voting Ensemble** | **~99.7%** | **~99.7%** | **~99.7%** | **~99.7%** |
| LSTM | ~69% | ~51% | ~71% | ~59% |

### Key Findings:
1. The **Voting Ensemble** achieves the best overall performance by combining the strengths of RF, MLP, and KNN.
2. **Random Forest** alone performs nearly as well as the ensemble, demonstrating the power of tree-based methods for tabular data.
3. **KNN** struggles with high-dimensional data and many classes — typical for this algorithm.
4. **LSTM** shows moderate performance for time-series forecasting, as predicting the exact crime type from temporal patterns alone is inherently challenging.

## 🔮 Future Improvements
- **Hyperparameter Tuning**: Use GridSearchCV or Optuna to optimize model parameters.
- **Feature Importance Analysis**: Identify which features contribute most to predictions.
- **Class Imbalance Handling**: Apply SMOTE or class weights to address underrepresented crime types.
- **Deep Learning for Tabular Data**: Experiment with TabNet or other modern architectures.
- **Real-time Deployment**: Integrate with the Flask web app for live predictions.

---

*This notebook was developed as a Final Year Project (FYP) — Chicago Crime Prediction System.*
